In [ ]:

#@title Run
Mount_Google_Drive = True #@param {type:"boolean"}

if Mount_Google_Drive:
    from google.colab import drive
    print("📁 Connecting to Google Drive...")
    drive.mount('/content/drive')

import os
import sys
import time
import base64
import urllib.request
import subprocess
import shutil

ENCODED_REPO = "aHR0cHM6Ly9naXRodWIuY29tL2F6YWRuZXR3b3JrL2R1YmJlci5naXQ="

REPO_URL = base64.b64decode(ENCODED_REPO.encode("utf-8")).decode("utf-8")
APP_DIR = "/content/dubbing-app"

subprocess.run("fuser -k 3000/tcp 2>/dev/null || true", shell=True)
subprocess.run("pkill -9 -f cloudflared 2>/dev/null || true", shell=True)

def is_node20_installed():
    try:
        res = subprocess.run(["node", "-v"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return res.returncode == 0 and res.stdout.strip().startswith("v20.")
    except:
        return False

if not is_node20_installed():
    print("⚙️ Setting up Node.js 20 & FFmpeg...")
    subprocess.run("apt-get purge -y nodejs npm libnode-dev gyp 2>/dev/null || true", shell=True)
    subprocess.run("apt-get autoremove -y 2>/dev/null || true", shell=True)
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null 2>&1", shell=True)
    subprocess.run("apt-get install -y nodejs ffmpeg build-essential >/dev/null 2>&1", shell=True)

if not shutil.which("yt-dlp"):
    subprocess.run("pip install -q -U yt-dlp", shell=True)

if not shutil.which("cloudflared"):
    subprocess.run("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True)

if not os.path.exists(APP_DIR):
    subprocess.run(f"git clone {REPO_URL} {APP_DIR}", shell=True)
else:
    subprocess.run(f"cd {APP_DIR} && git pull origin main 2>/dev/null || git pull", shell=True)

os.chdir(APP_DIR)

if not os.path.exists(os.path.join(APP_DIR, "node_modules")):
    print("📦 Installing dependencies...")
    subprocess.run("npm install", shell=True)

if not os.path.exists(os.path.join(APP_DIR, "dist")):
    print("🔨 Building project...")
    subprocess.run("npm run build", shell=True)

print("🚀 Starting production application server...")
env = os.environ.copy()
env["NODE_ENV"] = "production"

server_process = subprocess.Popen(
    ["npm", "start"],
    cwd=APP_DIR,
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

server_ready = False
for _ in range(60):
    try:
        req = urllib.request.Request("http://127.0.0.1:3000/api/health")
        with urllib.request.urlopen(req, timeout=2) as response:
            if response.status == 200:
                server_ready = True
                break
    except:
        pass
    time.sleep(1)

if not server_ready:
    print("❌ Server failed to respond to health check.")
    sys.exit(1)

print("🌐 Server is online! Launching Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url_printed = False
try:
    for line in iter(tunnel_process.stdout.readline, ''):
        if not tunnel_url_printed and "trycloudflare.com" in line:
            for part in line.split():
                if "trycloudflare.com" in part and part.startswith("http"):
                    print("\n" + "="*60)
                    print(f"✨ App is LIVE at: {part.strip()}")
                    print("="*60 + "\n")
                    tunnel_url_printed = True
                    break
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    tunnel_process.terminate()
    server_process.terminate()